# 02 NH투자증권(2018) 밸류 팩터 재현

## 0. 원문 정리
원문: NH투자증권 하재석·박소현(2018-07-16), 「Korea Factor Investing 시리즈(1): 밸류(Value) 팩터」. 시험 조건 상자(p.14), Multi-Metric(p.15), 그림 19(p.16), 리밸런싱 주기(p.18), 애널리스트 추정치(p.19~20), 시가총액·유동성(p.21~23), KOSDAQ(p.24), 현실적 밸류 투자(p.26).
회원 결정(2026-11-06): "원문에 말이 없으면 빼. 원문에 있으면 원문대로 해." 원문에 없는데 빼면 계산이 안 되는 칸은 ① 자료가 없거나 계산이 안 되면 그 종목을 그 달에서 뺀다 ② 나머지는 가장 단순한 쪽(12절 표 '이유' 칸에 '기본값'으로 적음).

| 항목 | 원문 | 쪽 | 우리 설정 |
|---|---|---|---|
| 유니버스 | KOSPI 상장종목 중 관리종목·거래정지 아닌 종목, 리밸런싱 당일 상장 종목(생존편향 제거) | 14 | `시장=["KS"]`, `("거래정지", {"아이템": CP10001650})`, `관리종목`(CP10001660), 그날 상장 가격 있는 종목. **보통주만**(원문에 없음, 회원 결정 2026-11-06) |
| 지표 | P/B(장부가), P/E(순이익), P/S(매출액), P/OCF(영업현금흐름), EV/EBITDA | 14 | `BP`·`EP`·`SP`·`CFP`·`EBITDA_EV` (역수 꼴, 높을수록 쌈) |
| 묶음 | 지표 기준 1~5분위, 동일가중 | 14 | `고르기=("분위",5)`, `비중=["동일"]` |
| 롱숏 | 1분위 Long − 5분위 Short. P/E·P/OCF는 적자 종목이 많아 1분위 − 4분위 | 14 | LS = Q1 − Q5 (P/E·P/OCF는 Q1 − Q4) |
| 리밸런싱 | 1개월 단위 월말 | 14 | `기준일="마지막거래일"`, `리밸런싱=1` |
| 재무 반영 | "주요 재무제표 수치는 분기/연간 재무제표가 발표되는 월의 월말에 반영" | 14 | `재무주기=("발표값",)` = `포.발표재무`: 기준일까지 발표된 가장 최근 보고서(1·2·3분기 보고서의 분기 3개월 값, 또는 사업보고서의 연간 값) 값 그대로(4분기 합·×4 같은 가공 없음, 회원 결정 2026-11-06). 쓸 수 있는 날 = 그 보고서 발표일(FP56000900) → 그 달(달력 달)에 발표된 보고서가 그 달 말 리밸런싱에 들어감. 영업CF·EBITDA 분기 3개월 값은 DataGuide가 누적값에서 환산·계산한 값(분기 현금흐름표는 누적으로 공시, EBITDA는 보고서에 없는 항목) |
| 수익률 가격 | "수익률을 동일가중으로 측정"뿐 | 14 | 수정주가 S410000700(배당 없음, 회원 결정 2026-11-06) |
| 기간 | 2002-03 ~ 2018-05 | 14 | `("2002-03","2018-05")` |
| 그림 16·17 | LS 누적(%), 그림 17은 2010년 이후 | 14·15 | 5절, 선형 % 축 |
| Multi-Metric | 5개 지표 Z-Score 합산 순위로 고른 1분위 Long Only 동일가중 | 15 | `합산="z합산"`, Q1 |
| 그림 18·19 | 1분위 누적(%), 월간 변동성 대비 절대수익률: Multi 0.39, P/B 0.34, P/E 0.32, EV/EBITDA 0.34, P/S 0.30, P/OCF 0.29 | 15·16 | 6절(선형 % 축), 10절(Q1 월평균 / 월 표준편차) |
| 그림 23·24 | P/B LS, 리밸런싱 1·3·6·12개월 | 18 | 7절 |
| 그림 25·26 | 애널리스트 추정 12개월 Forward P/E | 19·20 | 그림 25·26 칸(G02) |
| 그림 28·29 | KOSPI 전체·시총 상위 300위·100위 이내, P/B 1분위 Long, 5분위 Short / P/E 1분위 Long, 3분위 Short | 21·22 | 8절(P/E는 3분위 = 3그룹으로 읽음 — 해석: 그림 29에 분위 수가 따로 없고, p.19가 3그룹을 그림 25 제목에서 '1분위 Long, 3분위 Short'라 부름. 다른 읽기와 버린 이유는 12절) |
| 그림 30·31 | 월평균 거래대금 10억·20억 원 이상, KOSPI 전체·시총 300위 이내, P/B 1분위 − 5분위 | 23 | 8절 |
| 그림 32·33 | KOSDAQ 시총 300위 이내 상위 1/3 − 하위 1/3(P/B·P/E), KOSPI 시총 300위 P/B 상위 1/3 − 하위 1/3와 비교 | 24 | 8절 |
| 현실판(그림 36) | KOSPI 시총 300위 이내, 월평균 거래대금 10억 원 이상, Multi-Metric, 매매비용 월 0.2%, 벤치마크 KOSPI 200 동일가중지수, 가로축 '09.1~'18.1 | 26 | `("시총상위",{"n":300})`, `("거래대금", {"창": "월"})`(기준일이 든 달의 거래일 평균), `거래비용=dict(..., 월비용=0.002)`, 벤치마크 `I.401`. 2008-12 말 첫 리밸런싱(첫 수익률 2009-01, 원문 그림 '09.1 첫 점이 0 아래) ~ 2018-05(끝 달은 원문에 없어 본 실험 끝). I.401 값이 2009-01-02부터라 I.401 수익률과 대조는 2009-02부터(자료 한계) |

줄임말: LS(롱숏: 좋은 분위를 사고 나쁜 분위를 판 수익률), EW(동일가중), MDD(최대 낙폭), IR(정보비율), CAPM(자본자산가격결정모형), FF3(파마-프렌치 3요인), Carhart(카하트 4요인), OLS(보통최소제곱).

## 1. 불러오기
계산은 모두 `공통/` 폴더의 모듈이 한다. 이 노트북은 설정을 정하고, 실행하고, 결과를 본다.

In [ ]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "공통"))
import numpy as np
import pandas as pd
import 데이터읽기 as 읽기, 종목거르기, 지표, 포트폴리오 as 포, 평가
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
# 짧은 시험(환경변수 FOLLOW_TEST=1): 기간을 첫 달부터 약 2년으로 줄이고(2절), 결과 파일은 _검증\1106\시험결과\에 쓴다(결과_월수익률\ 파일을 덮어쓰지 않음)
시험 = os.environ.get("FOLLOW_TEST") == "1"
if 시험:
    평가.결과폴더 = Path.cwd().parent / "_검증" / "1106" / "시험결과"
    평가.결과폴더.mkdir(parents=True, exist_ok=True)

## 2. 데이터 경로와 기간
데이터를 어디서 읽을지, 어느 기간을 돌릴지 정한다. 기간은 (첫 리밸런싱 월, 마지막 기준일 월)이다.

In [ ]:
# 새로 통째로 받은 파일(읽기.DATA 기본 = data\통합_전체_202611, DataGuide 달력기준 일간)로 원문 기간을 돌린다.
기간 = ("2002-03", "2018-05")  # 원문 p.14 '시뮬레이션 기간: 2002년 3월~ 2018년 5월'
현실기간 = ("2008-12", 기간[1])  # 원문 p.26 그림 36 가로축 '09.1~'18.1, '09.1 첫 점이 이미 0 아래 → 2008-12 말 첫 리밸런싱, 첫 수익률 2009-01(끝 달은 원문에 없어 본 실험 끝)
if 시험:  # 짧은 시험: 각 기간 첫 달부터 약 2년
    기간, 현실기간 = ("2002-03", "2004-02"), ("2008-12", "2010-11")
print("데이터:", 읽기.DATA, "/ 기간:", 기간, "/ 현실판 기간:", 현실기간, "/ 시험:", 시험)

## 3. 설정
지표만 바꿔 같은 설정으로 돌린다.
- 재무: 원문 p.14 "주요 재무제표 수치는 분기/연간 재무제표가 발표되는 월의 월말에 반영" → 기준일까지 발표된 가장 최근 보고서 값 그대로(`포.발표재무`, 회원 결정 2026-11-06: 4분기 합·×4 같은 가공 없음). 발표일이 없는 보고서는 쓰지 않는다(기본값 ①). 오래된 보고서 상한은 두지 않는다(기본값 ②).
- 보통주만, 수익률은 수정주가(회원 결정 2026-11-06, 원문에 없음).
- 원문 본 실험 조건(p.14)에는 거래비용 언급이 없어 0으로 둔다(기본값 ②; 원문이 비용을 뺐는지는 알 수 없음).

In [ ]:
# 재무시점은 발표값에서는 쓰지 않는다(보고서 발표일 = 쓸 수 있는 날, 포.발표행). '보통주'는 관리·정지 다음(현실판 시총 300위 순위도 보통주 안에서 매김)
기본 = {**포.기본설정, "이름": "NH밸류", "시장": ["KS"], "가격": 읽기.수정주가, "기간": 기간, "기준일": "마지막거래일", "리밸런싱": 1,
       "재무주기": ("발표값",), "재무시점": ("발표월말", 0), "제외": [("거래정지", {"아이템": 읽기.정지여부}), "관리종목", "보통주"],
       "고르기": ("분위", 5), "비중": ["동일"], "거래비용": dict(수수료=0, 거래세=0, 슬리피지=0)}
지표명 = {"P/B": "BP", "P/E": "EP", "P/S": "SP", "P/OCF": "CFP", "EV/EBITDA": "EBITDA_EV"}
숏 = {"P/E": "Q4", "P/OCF": "Q4"}  # 원문 p.14: 적자 종목이 많아 1분위 − 4분위
pd.Series(기본).astype(str).to_frame("공통 설정")

## 4. 실행: 지표별 5분위 + Multi-Metric

In [ ]:
결과 = {n: 포.실행({**기본, "지표": [(m, "높게")]}) for n, m in 지표명.items()}
결과["Multi-Metric"] = 포.실행({**기본, "지표": [(m, "높게") for m in 지표명.values()], "합산": "z합산"})

## 5. 롱숏(LS) 누적 (그림 16·17)
원문 그림 16(p.14)·17(p.15): 세로축 누적 수익률(%), 선형 축. 그림 17은 2010년 이후(2009-12 말을 0으로 두고 2010-01 수익률부터 다시 누적). 범례 순서는 원문대로.

In [ ]:
LS = pd.DataFrame({n: o["수익률"]["동일"]["Q1"] - o["수익률"]["동일"][숏.get(n, "Q5")] for n, o in 결과.items() if n != "Multi-Metric"})
원문순서 = ["P/B", "EV/EBITDA", "P/E", "P/S", "P/OCF"]  # 원문 그림 16·17 범례 순서
평가.누적그림(LS[원문순서], "그림 16. 측정지표에 따른 KOSPI 밸류 프리미엄(1분위 Long, 5분위 Short; P/E·P/OCF는 4분위 Short)", 퍼센트=True);
if len(LS.loc["2010":]):  # 짧은 시험 기간에는 2010년 이후가 없음
    평가.누적그림(LS.loc["2010":, 원문순서], "그림 17. 측정지표에 따른 KOSPI 밸류 프리미엄(2010년 이후)", 퍼센트=True);

## 6. 1분위 Long Only 누적 (그림 18)
원문 그림 18(p.15): 1분위 동일가중 누적 수익률(%), 선형 축, Multi-Metric과 5개 지표.

In [ ]:
Q1 = pd.DataFrame({n: o["수익률"]["동일"]["Q1"] for n, o in 결과.items()})
평가.누적그림(Q1[["Multi-Metric", *원문순서]].dropna(how="all"), "그림 18. 측정지표에 따른 1분위 종목 동일가중 수익률", 퍼센트=True);

## 7. 리밸런싱 주기 (그림 23·24, p.18)
원문 p.18: "KOSPI 상장종목을 대상으로 P/B 지표를 사용해서 앞서와 동일한 로직으로 1개월, 3개월, 6개월, 12개월로 리밸런싱 주기를 변경". 그림 23 = 네 주기, 그림 24 = 3·6·12개월(P/B 1분위 Long, 5분위 Short, 누적 %).
1개월은 4절 P/B 결과와 같은 설정이라 그대로 쓴다. 3·6·12개월은 첫 리밸런싱(2002-03 말)부터 k달마다 다시 고르고 사이에는 매수보유(시작 달은 원문에 없음, 기본값 ②).

In [ ]:
주기 = {"1개월": 결과["P/B"]} | {f"{k}개월": 포.실행({**기본, "지표": [("BP", "높게")], "리밸런싱": k}) for k in (3, 6, 12)}
주기LS = pd.DataFrame({k: o["수익률"]["동일"]["Q1"] - o["수익률"]["동일"]["Q5"] for k, o in 주기.items()})
평가.누적그림(주기LS, "그림 23. P/B 지표 기준 1분위 Long, 5분위 Short 포트폴리오 성과(KOSPI)", 퍼센트=True);
평가.누적그림(주기LS.drop(columns="1개월"), "그림 24. 리밸런싱 주기가 길어질수록 효과가 크게 감소", 퍼센트=True);

## 애널리스트 추정치 P/E (원문 p.19 그림 25, p.20 그림 26)
원문 p.19 "2) 애널리스트 추정치 사용": "이에 동일한 로직으로 확정치가 아닌 애널리스트의 최근 3개월 EPS 추정치를 사용해 매월 리밸런싱하는 밸류 포트폴리오의 성과를 점검했다.
3개월 이내 애널리스트 추정치가 존재하는 종목의 수가 대략 300개 정도로 총 3개 그룹으로 구분해 상위 1/3 Long, 하위 1/3 Short 포트폴리오의 성과로 분석했다."

| 항목 | 원문 | 쪽 | 우리 |
|---|---|---|---|
| 추정치 | 애널리스트의 최근 3개월 EPS 추정치 | 19 | DataGuide 컨센서스 '3개월' EPS(지배) FY1·FY2(`읽기.컨센서스`, FM30011505 R17·R18) |
| 지표 | 그림 25·26 범례 '12개월 Fwd P/E' | 19·20 | 역수 꼴 `EP_선행` = 12개월 선행 EPS / 수정주가(본문 P/E도 역수 꼴 E/P라 같은 꼴) |
| 12개월 선행 계산 | 원문에 없음 | — | FY1 × (12 − m)/12 + FY2 × m/12(`지표.EP_선행`). 우리가 고른 값 |
| 묶음 | 3개 그룹, 상위 1/3 Long·하위 1/3 Short(그림 25 제목 '1분위 Long, 3분위 Short') | 19 | `고르기=("분위", 3)`, LS = Q1 − Q3 |
| 유니버스 | '동일한 로직' + 3개월 이내 추정치가 있는 종목 | 19 | 본문 `기본` 거르기 뒤 FY1·FY2 추정치가 모두 있는 종목(없으면 그 달 제외). 원문 조건(p.19 "3개월 이내 애널리스트 추정치가 존재하는 종목") |
| 리밸런싱 | 매월 | 19 | 본문과 같음(월말, 1개월) |
| 그림 25 기간 | 가로축 '10.1~'18 | 19 | 2010-01 말 첫 리밸런싱 ~ 본문 끝(7절 현실판 '09.1과 같은 읽기). 본문 실행에서 2010-02 수익률부터 자름(매월 리밸런싱·비용 0이라 2010-01부터 따로 돌린 것과 같음) |
| 그림 26 기간 | "2002년 이후", 가로축 '02.4~ | 19·20 | 본문 기간 |
| 그림 26 묶음 | 상위 1/3 Long Only 동일가중 | 19 | Q1, 동일가중 |
| 비교선 'P/E' | 확정 P/E(그림 25·26 범례 'P/E') | 19·20 | 본문 `EP`(발표 재무), 같은 3분위. 유니버스는 본문 전체(원문 p.19 "추정치가 존재하는 종목 수가 제한적"이 확정 P/E 쪽이 더 넓다는 뜻으로 읽음) |

원문 판정(방향·크기만): ① 그림 25 "P/E 지표를 이용한 밸류 포트폴리오가 2014년 이후 성과가 부진했던 것과 달리 애널리스트 추정치 P/E를 사용할 경우 꾸준히 효과적이었다"(p.19)
② 그림 26 "2002년 이후 상위 1/3 Long Only 동일가중 포트폴리오 누적 수익률에서 확정 P/E를 사용하는 데에 비해 크게 못 미쳤다"(p.19) ③ "3개월 이내 애널리스트 추정치가 존재하는 종목의 수가 대략 300개"(p.19). ③의 판정 잣대(리밸런싱마다 종목 수 평균을 백 단위로 반올림해 300)는 원문에 없어 우리가 고른 값.

In [ ]:
# 시험(FOLLOW_TEST=1)이면 1절에서 평가.결과폴더가 _검증\1106\시험결과\로 바뀌어 있음
추정 = 포.실행({**기본, "지표": [("EP_선행", "높게")], "고르기": ("분위", 3)})
확정 = 포.실행({**기본, "지표": [("EP", "높게")], "고르기": ("분위", 3)})
q3 = lambda o: o["수익률"]["동일"]
LS3 = pd.DataFrame({"12개월 Fwd P/E": q3(추정)["Q1"] - q3(추정)["Q3"], "P/E": q3(확정)["Q1"] - q3(확정)["Q3"]})
Q1_3 = pd.DataFrame({"12개월 Fwd P/E": q3(추정)["Q1"], "P/E": q3(확정)["Q1"]})
추정수 = 추정["기록"][["편입:Q1", "편입:Q2", "편입:Q3"]].sum(axis=1)  # 그 달 추정치가 있어 3분위에 들어간 종목 수
print("추정치 종목 수(리밸런싱마다):", 추정수.describe().round(1).to_dict(), "/ 첫 리밸런싱", f"{추정수.index[0]:%Y-%m-%d}", int(추정수.iloc[0]))
시작25 = "2010-02" if not 시험 else LS3.index[0]  # 시험(짧은 기간)은 2010년 전이라 전체 구간
LS25 = LS3.loc[시작25:]
평가.누적그림(LS25, "그림 25: 12개월 Fwd P/E와 P/E, 1분위 Long·3분위 Short", 퍼센트=True);
평가.누적그림(Q1_3.dropna(how="all"), "그림 26: 12개월 Fwd P/E와 P/E, 상위 1/3 Long Only 동일가중", 퍼센트=True);

In [ ]:
누25 = (1 + LS25).prod() - 1
후 = LS25.loc["2014":]  # 원문 '2014년 이후'
누후 = (1 + 후).prod() - 1
누26 = (1 + Q1_3).prod() - 1
대조25_26 = pd.DataFrame([
    ["p.19 그림 25", "애널리스트 추정치 P/E LS는 꾸준히 효과적", "(우리가 읽은 기준) 12개월 Fwd P/E LS 누적 > 0",
     f"Fwd {누25['12개월 Fwd P/E']:.3f} ({LS25.index[0]:%Y-%m}~{LS25.index[-1]:%Y-%m})", "같음" if 누25["12개월 Fwd P/E"] > 0 else "다름"],
    ["p.19 그림 25", "P/E는 2014년 이후 부진, 추정치 P/E는 달리 효과적", "(우리가 읽은 기준) 2014-01~ LS 누적: Fwd > 0이고 Fwd > P/E",
     f"2014~ Fwd {누후['12개월 Fwd P/E']:.3f}, P/E {누후['P/E']:.3f}" if len(후) else "구간 없음(시험 기간)",
     ("같음" if 누후["12개월 Fwd P/E"] > 0 and 누후["12개월 Fwd P/E"] > 누후["P/E"] else "다름") if len(후) else "판정 안 함"],
    ["p.19 그림 26", "상위 1/3 Long Only 누적: 추정치 P/E가 확정 P/E에 크게 못 미침", "Fwd Q1 누적 < P/E Q1 누적(크기는 두 누적을 그대로 적음)",
     f"Fwd {누26['12개월 Fwd P/E']:.3f}, P/E {누26['P/E']:.3f} ({Q1_3.dropna(how='all').index[0]:%Y-%m}~{Q1_3.index[-1]:%Y-%m})",
     "같음" if 누26["12개월 Fwd P/E"] < 누26["P/E"] else "다름"],
    ["p.19", "3개월 이내 추정치가 있는 종목 수 대략 300개", "(우리가 읽은 기준) 리밸런싱마다 종목 수 평균을 백 단위로 반올림하면 300",
     f"평균 {추정수.mean():.1f}, 최소 {추정수.min()}, 최대 {추정수.max()}", "같음" if round(추정수.mean(), -2) == 300 else "다름"],
], columns=["원문 쪽·그림", "원문 결론", "판정 기준", "우리", "판정"])
대조25_26.to_csv(평가.결과폴더 /"02_원문대조_그림25_26.csv", encoding="utf-8-sig")
대조25_26

## 8. 시가총액별·유동성별·KOSDAQ (그림 28~33, p.21~24)
- 그림 28·29(p.21~22): "KOSPI 전체 종목, 대형주와 중형주를 포함하는 시가총액 상위 300위 종목 및 대형주만을 포함하는 시가총액 상위 100위 이내 종목을 대상으로 P/B, P/E 기준 성과를 검증". 그림 28 주 "P/B 기준 1분위 Long, 5분위 Short", 그림 29 주 "P/E 기준 1분위 Long, 3분위 Short" → 3분위(3그룹) Q1 − Q3로 읽음(해석): 같은 보고서 p.19 "총 3개 그룹으로 구분해 상위 1/3 Long, 하위 1/3 Short"를 그림 25 제목이 "1분위 Long, 3분위 Short"라고 부름(분위 수는 그림 29에 따로 없음, `_검증/1106/원문사양_02.md` 3절 1). 세 유니버스마다 그 안에서 분위를 새로 나눔.
- 그림 30·31(p.23): "월평균 거래대금 10억원 이상, 20억원 이상인 종목으로 한정", 그림 30 = KOSPI 전체, 그림 31 = 시가총액 300위 이내, 둘 다 "P/B 기준 1분위 Long, 5분위 Short". '월평균'은 현실판(9절)과 같게 기준일이 든 달의 거래일 평균 거래대금.
- 그림 32·33(p.24): "적자기업 등이 많은 KOSDAQ 특성상 시가총액 300위 이내 기업의 밸류 지표 상위 1/3 – 하위 1/3 동일가중 수익률", 그림 33 주 "KOSPI, KOSDAQ 시가총액 상위 300위 종목 대상 P/B 기준 상위 1/3 Long, 하위 1/3 Short" → 3분위 Q1 − Q3.
- 나머지(관리·정지 제외, 보통주, 월말 1개월 리밸런싱, 재무 발표값, 기간)는 본 실험과 같음(원문 p.18 "앞서와 동일한 로직"). 시총 순위는 관리·정지·보통주를 거른 뒤 남은 종목에서 매긴다(지표가 있는지 보기 전, 원문에 없음 → 기본값 ②). 시총과 거래대금을 둘 다 거를 때는 시총 먼저(현실판과 같음, 원문에 순서 없음 → 기본값 ②).

`LS판`은 같은 거르기로 P/B·P/E를 한 번에 돌린다(`포.실행`의 `전략들`: 같은 날 같은 남은 종목에서 지표마다 따로 줄 세움 = 지표마다 따로 돌린 것과 같은 계산).

In [ ]:
def LS판(숏들, 더제외=(), 시장=("KS",), 분위=5):
    """본 실험 설정에 거르기를 더해 지표마다 1분위 − 숏 분위 월수익률. 숏들 = {지표명 키: 숏 분위}."""
    R = 포.실행({**기본, "시장": list(시장), "제외": 기본["제외"] + list(더제외), "고르기": ("분위", 분위),
                 "전략들": {n: dict(지표=[(지표명[n], "높게")], 합산="순위합산") for n in 숏들}})["수익률"]
    return pd.DataFrame({n: R[(n, "동일", "Q1")] - R[(n, "동일", q)] for n, q in 숏들.items()})
시총 = lambda n: ("시총상위", {"n": n})
대금 = lambda 하한: ("거래대금", {"하한": 하한, "창": "월"})
시총300, 시총100 = LS판({"P/B": "Q5"}, [시총(300)]), LS판({"P/B": "Q5"}, [시총(100)])  # 그림 28: 5분위
셋300 = LS판({"P/B": "Q3", "P/E": "Q3"}, [시총(300)], 분위=3)  # 3분위: 그림 29의 300위 선(P/E), 그림 33의 KOSPI 선(P/B)
그림28 = pd.DataFrame({"KOSPI 전체": LS["P/B"], "KOSPI 시총300위 이내": 시총300["P/B"], "KOSPI 시총100위 이내": 시총100["P/B"]})
그림29 = pd.DataFrame({"KOSPI 전체": LS판({"P/E": "Q3"}, 분위=3)["P/E"], "KOSPI 시총300위 이내": 셋300["P/E"],
                     "KOSPI 시총100위 이내": LS판({"P/E": "Q3"}, [시총(100)], 분위=3)["P/E"]})
평가.누적그림(그림28, "그림 28. KOSPI도 시가총액이 낮을수록 밸류 프리미엄이 높아(P/B 기준, 1분위 Long, 5분위 Short)", 퍼센트=True);
평가.누적그림(그림29, "그림 29. KOSPI 시가총액 낮을수록 밸류 프리미엄이 높아(P/E 기준, 1분위 Long, 3분위 Short)", 퍼센트=True);

In [ ]:
그림30 = pd.DataFrame({"거래대금 조건X": LS["P/B"], "10억원 이상": LS판({"P/B": "Q5"}, [대금(1e9)])["P/B"], "20억원 이상": LS판({"P/B": "Q5"}, [대금(2e9)])["P/B"]})
그림31 = pd.DataFrame({"거래대금 조건X": 시총300["P/B"], "10억원 이상": LS판({"P/B": "Q5"}, [시총(300), 대금(1e9)])["P/B"],
                     "20억원 이상": LS판({"P/B": "Q5"}, [시총(300), 대금(2e9)])["P/B"]})
평가.누적그림(그림30, "그림 30. 유동성이 낮을수록 밸류 프리미엄이 높아(KOSPI 전체 대상 P/B 기준)", 퍼센트=True);
평가.누적그림(그림31, "그림 31. 대형주, 중형주의 유동성과 밸류 프리미엄(KOSPI 시가총액 300위 이내 대상 P/B 기준)", 퍼센트=True);

In [ ]:
그림32 = LS판({"P/B": "Q3", "P/E": "Q3"}, [시총(300)], 시장=("KQ",), 분위=3)
그림33 = pd.DataFrame({"KOSDAQ": 그림32["P/B"], "KOSPI": 셋300["P/B"]})  # 원문 그림 33 주: KOSPI·KOSDAQ 시총 300위, P/B 상위 1/3 − 하위 1/3
평가.누적그림(그림32, "그림 32. KOSDAQ 종목에서도 밸류 프리미엄이 확인됨(시총 300위 이내, 상위 1/3 Long, 하위 1/3 Short)", 퍼센트=True);
평가.누적그림(그림33, "그림 33. 2002년 이후 KOSDAQ의 밸류 프리미엄이 KOSPI보다 높아(P/B 기준, 시총 300위 이내)", 퍼센트=True);

## 9. 현실판(p.26): 시총 300위 이내 + 거래대금 10억 원 이상, Multi-Metric, 매매비용 월 0.2%
원문(p.26): "이러한 방식으로 매월 전체 포트폴리오의 0.2% 매매비용을 감안해 1개월 단위 리밸런싱한 결과 KOSPI 200 동일가중지수를 Outperform할 수 있었다(그림36 참조)."
그림 36 제목(p.26): "5가지 지표 Multi-Metric 방식으로 선정한 동일가중 포트폴리오 수익률 (1개월 단위 리밸런싱, 매매비용 월 0.2% 가정)".
본 결과는 매달 포트폴리오의 0.2%를 뺀 판이다(그림 36: 세로축 누적 %, 선형). 기간: 원문 그림 36 '09.1 첫 점이 0 아래라 2008-12 말 첫 리밸런싱(첫 수익률 2009-01). I.401은 값이 2009-01-02부터라 그림 36의 I.401 선과 Outperform 대조는 2009-02부터(자료 한계). 벤치마크는 원문대로 KOSPI 200 동일가중지수(X01 `I.401` '코스피 200 동일가중지수', 종가지수, 값은 2009-01-02부터)를 놓는다(2026-11-04 고침, 예전엔 '데이터에 없음'으로 잘못 보고 코스피 I.001). 배당 포함 여부는 X01에 적혀 있지 않다(확인 안 함). 포트폴리오는 수정주가(배당 없음).

In [ ]:
# 기간 = 현실기간(2절: 2008-12 말 첫 리밸런싱 ~ 2018-05). 유니버스: 시총 300위·월평균 거래대금 10억
현실설정 = {**기본, "이름": "NH밸류_현실판", "기간": 현실기간, "지표": [(m, "높게") for m in 지표명.values()], "합산": "z합산",
           "제외": 기본["제외"] + [("시총상위", {"n": 300}), ("거래대금", {"창": "월"})]}
월비용 = dict(수수료=0, 거래세=0, 슬리피지=0, 월비용=0.002)
현실 = {"현실판_비용월0.2%(본)": 포.실행({**현실설정, "거래비용": 월비용})}
F200 = 평가.요인(포.기준일({**기본, "기간": (기간[0], 현실기간[1])}), {**기본, "시장지수": "I.401"})  # 원문 벤치마크(p.26) KOSPI 200 동일가중지수
현실표 = pd.DataFrame({k: o["수익률"][("동일", "Q1")] for k, o in 현실.items()} | {"KOSPI200 동일가중(I.401)": F200["시장"]})
본36 = ["현실판_비용월0.2%(본)", "KOSPI200 동일가중(I.401)"]
그림36 = 현실표.loc["2009-02":, 본36].dropna().set_axis(["밸류 포트폴리오(비용 월 0.2%)", "KOSPI 200 동일가중지수(I.401)"], axis=1)  # 대조용: 두 선 모두 값 있는 달(2009-02~)
평가.누적그림(현실표.loc[현실표[본36[0]].first_valid_index():, 본36].set_axis(["밸류 포트폴리오(비용 월 0.2%, 2009-01~)", "KOSPI 200 동일가중지수(I.401, 2009-02~)"], axis=1),
         "그림 36. 5가지 지표 Multi-Metric 동일가중 포트폴리오(1개월 리밸런싱, 매매비용 월 0.2%)", 퍼센트=True);

## 10. 원문 vs 우리
원문은 대부분 그림이라 숫자는 그림 19(위험조정수익률)만 비교한다. 나머지는 원문의 문장(주장)과 우리 결과가 같은 방향인지 본다.

In [ ]:
그림19 = pd.Series({"Multi-Metric": 0.39, "P/B": 0.34, "P/E": 0.32, "EV/EBITDA": 0.34, "P/S": 0.30, "P/OCF": 0.29})
우리19 = (Q1.mean() / Q1.std())[그림19.index]
그림19표 = pd.DataFrame({"원문 그림19": 그림19, "우리(Q1 월평균/월표준편차)": 우리19})
display(그림19표)
누적LS = (1 + LS).prod() - 1
주장 = pd.DataFrame({"원문 주장": ["모든 지표에서 LS 누적이 양(+)", "P/B·P/E가 가장 효과적(LS 누적 상위 2)", "Multi-Metric 1분위가 단일 지표 1분위보다 위험조정수익률 높음"],
              "우리": [bool((누적LS > 0).all()), sorted(누적LS.nlargest(2).index) == ["P/B", "P/E"], bool(우리19.idxmax() == "Multi-Metric")]})
그림19표.to_csv(평가.결과폴더 / "02_원문대조_그림19.csv", encoding="utf-8-sig")  # 결과요약 노트북이 읽음
주장.to_csv(평가.결과폴더 / "02_원문주장.csv", encoding="utf-8-sig")
주장

### 원문 결론 판정 (원문이 말한 방향·크기만)
원문 문장마다 원문이 말한 방향·크기만 본다(p값 기준 없음). '판정 기준' 칸은 숫자 기준이 없는 원문 말을 우리가 읽은 방법이다. 그림 31 "감소 폭이 제한적"의 잣대(조건X 대비 20억 원 이상 LS 누적 감소율을 그림 30과 비교)는 원문에 없어 우리가 고른 값. 그림 25·26(애널리스트 추정치) 결론은 G02 칸.

원문 안에서 서로 어긋나는 곳(원문 그대로 적음, 그림 값은 눈대중, `_검증/1106/원문사양_02.md`):
- p.14 본문은 "P/E와 P/OCF 지표의 경우 … 1분위 Long, 4분위 Short"인데 그림 16 제목은 "(1분위 Long, 5분위 Short)". 우리는 본문을 따름.
- P/B LS 누적 끝값이 그림 16에서는 약 2,250%, 같은 P/B 1개월 LS인 그림 23·28·30에서는 약 850%.
- 그림 24 제목은 "리밸런싱 주기가 길어질수록 효과가 크게 감소"인데 그림에서는 6개월(약 25%)이 12개월(약 45%)보다 낮게 끝남.
- p.14는 "P/E는 2015년 이후 다소 부진", p.19는 "P/E 지표를 이용한 밸류 포트폴리오가 2014년 이후 성과가 부진". 판정은 p.14 문장(그림 16)대로 2015년으로 나눔.
- 그림 31에서는 10억 원 이상(약 530%)이 조건 X(약 460%)보다 높게 끝남.

In [ ]:
누적 = lambda x: (1 + x).prod() - 1
연환산 = lambda x: (1 + x.dropna()).prod() ** (12 / x.dropna().size) - 1 if x.dropna().size else np.nan
판 = lambda ok: "같음" if ok else "다름"
줄 = lambda s: ", ".join(f"{k} {v:.3f}" for k, v in s.items())
누적Q1 = 누적(Q1[list(결과)])
최고단일 = 누적Q1.drop("Multi-Metric").idxmax()
ls10 = 누적(LS.loc["2010":])
pe전, pe후 = 연환산(LS["P/E"].loc[:"2014"]), 연환산(LS["P/E"].loc["2015":])
주기누적 = 누적(주기LS)
그림누적 = {k: 누적(v) for k, v in dict(그림28=그림28, 그림29=그림29, 그림30=그림30, 그림31=그림31, 그림32=그림32, 그림33=그림33, 그림36=그림36).items()}
감소율 = lambda s: (s["거래대금 조건X"] - s["20억원 이상"]) / s["거래대금 조건X"]  # 거래대금 20억 거르기로 줄어든 LS 누적의 비율
결론 = pd.DataFrame([
    ["p.15 그림 18", "Multi-Metric 1분위 누적이 단일 지표 1분위보다 '월등하게 우수'", "Multi 1분위 누적 > 단일 지표 1분위 누적 최고('월등'의 크기 기준은 원문에 없음)",
     f"Multi {누적Q1['Multi-Metric']:.3f}, 단일 최고 {최고단일} {누적Q1[최고단일]:.3f}", 판(누적Q1.idxmax() == "Multi-Metric")],
    ["p.14 그림 17", "2010년 이후로 한정해도 P/B·P/E가 가장 효과적", "2010-01~ LS 누적 상위 2 = P/B·P/E", 줄(ls10.sort_values(ascending=False)), 판(set(ls10.nlargest(2).index) == {"P/B", "P/E"})],
    ["p.14 그림 16", "P/E는 2015년 이후 다소 부진", "P/E LS 연환산: 2015-01~ < ~2014-12", f"~2014 {pe전:.4f}, 2015~ {pe후:.4f}", 판(pe후 < pe전)],
    ["p.18 그림 23·24", "P/B LS는 1개월 리밸런싱이 가장 우수(1·3·6·12개월)", "LS 누적이 1개월에서 가장 큼", 줄(주기누적), 판(주기누적.idxmax() == "1개월")],
    ["p.21 그림 28", "P/B LS 누적: KOSPI 전체 > 시총 300위 이내 > 시총 100위 이내", "세 LS 누적이 이 순서", 줄(그림누적["그림28"]), 판(그림누적["그림28"].is_monotonic_decreasing)],
    ["p.21~22 그림 29", "P/E LS(1분위 − 3분위) 누적: KOSPI 전체 > 시총 300위 이내 > 시총 100위 이내", "세 LS 누적이 이 순서", 줄(그림누적["그림29"]), 판(그림누적["그림29"].is_monotonic_decreasing)],
    ["p.23 그림 30", "거래대금 10억·20억 원 이상으로 한정하면 밸류 프리미엄 감소(유동성 낮을수록 높음)", "KOSPI 전체 P/B LS 누적: 조건X > 10억 > 20억", 줄(그림누적["그림30"]), 판(그림누적["그림30"].is_monotonic_decreasing)],
    ["p.23 그림 31", "시총 300위 이내 대상이면 감소 폭이 제한적", "(조건X − 20억) / 조건X LS 누적 감소율이 시총 300위(그림 31) < KOSPI 전체(그림 30)('제한적' 크기 기준은 원문에 없음)",
     f"감소율 전체 {감소율(그림누적['그림30']):.3f}, 시총300 {감소율(그림누적['그림31']):.3f}", 판(감소율(그림누적["그림31"]) < 감소율(그림누적["그림30"]))],
    ["p.24 그림 32", "KOSDAQ에도 밸류 프리미엄이 존재(P/B·P/E)", "KOSDAQ 시총 300위 P/B·P/E 3분위 LS 누적 > 0", 줄(그림누적["그림32"]), 판((그림누적["그림32"] > 0).all())],
    ["p.24 그림 33", "KOSPI 대비 오히려 KOSDAQ의 밸류 프리미엄이 큼(P/B)", "시총 300위 P/B 3분위 LS 누적: KOSDAQ > KOSPI", 줄(그림누적["그림33"]), 판(그림누적["그림33"]["KOSDAQ"] > 그림누적["그림33"]["KOSPI"])],
    ["p.26 그림 36", "현실판(비용 월 0.2%)이 KOSPI 200 동일가중지수를 Outperform", "같은 달 누적: 밸류 포트폴리오 > I.401", 줄(그림누적["그림36"]),
     판(그림누적["그림36"].iloc[0] > 그림누적["그림36"].iloc[1])],
], columns=["원문 쪽·그림", "원문 결론", "판정 기준", "우리", "판정"])
결론.to_csv(평가.결과폴더 / "02_원문결론.csv", encoding="utf-8-sig")  # 원문 대조 저장
결론

## 11. 차이 이유 후보
- **재무 값**: 원문 p.14대로 발표된 보고서를 발표 월 월말에 반영하고, 값은 발표된 값 그대로 쓴다(회원 결정 2026-11-06). 그래서 같은 달에 분기 보고서(3개월 순이익·매출·영업CF·EBITDA)를 낸 종목과 아직 사업보고서(12개월 값)만 있는 종목이 한 줄에 섞인다(주로 2~4월 사업보고서 제출 기간, 그리고 12월 결산이 아닌 회사는 다른 달에도. KS 보통주 실측: 2002-03-29 분기 152·연간 356, 2009-03 분기 71·연간 609, 2012-02 연간 56·분기 671; V02 검토, 휴일 규칙 고치기 전 숫자). 원문이 흐름 항목을 어떻게 했는지는 원문에 없다. 영업CF·EBITDA의 분기 3개월 값은 보고서 숫자 그대로가 아니라 DataGuide가 환산·계산한 값이다.
- **지표 방향**: 원문은 P/B 등 배수 이름을 쓴다. 우리는 역수(B/P 등)로 줄 세워 적자·자본잠식 종목이 '싼 쪽'에 섞이지 않는다(정렬 꼴은 원문에 없음). 원문이 P/E·P/OCF를 Q1−Q4로 한 것은 적자 종목이 Q5에 모이는 것과 같은 효과로 보인다.
- **Z-Score**: 원문의 극단값 처리(윈저라이즈 등)가 나와 있지 않다. 우리는 처리하지 않는다(기본값 ②). 5개 지표가 모두 있는 종목만 쓴다(기본값 ①).
- **EV·EBITDA 정의**: 원문에 정의가 없다. 우리는 EV = 시가총액 + 부채총계 − 현금(회원 결정 2026-11-04), EBITDA = DataGuide M000701032(기본값 ②).
- **현실판 벤치마크**: KOSPI 200 동일가중지수(I.401, 종가지수, 배당 포함 여부는 X01에 없음). 포트폴리오는 수정주가(배당 없음).

## 12. 원문과 다르게 한 것·원문에 없어 정한 것
| 항목 | 원문 | 우리 | 이유 |
|---|---|---|---|
| 재무 | 분기/연간 재무제표가 발표되는 월의 월말에 반영(p.14) | 기준일까지 발표된 가장 최근 보고서 값 그대로(`포.발표재무`). 쓸 수 있는 날 = 보고서 발표일(FP56000900), 그 달(달력 달)에 발표된 것은 마지막 거래일 뒤 같은 달 휴일 발표분까지 그 달 말에 씀(원문 문장대로, 그 휴일 발표분은 리밸런싱일 종가에는 몰랐던 값 = 최대 2일 미래 정보(전체 1,749행, 주로 2007-03·2002-03·2001-03), 한계). 같은 까닭으로 마지막 거래일 장 마감 뒤 발표분도 그날 종가에 씀 = 몇 시간 미래 정보(원문 문장대로, 한계; 발표 시각 자료가 없어 건수는 못 셈). 같은 날 두 보고서면 기간 끝이 늦은 것. 발표일이 없는 보고서는 안 씀. 오래된 보고서 상한 없음 | 원문 문장대로(p.14 "발표되는 월의 월말에 반영": 그 달 마지막 거래일 발표분·그 뒤 같은 달 휴일 발표분도 그 달 말에 씀 = 원문에 있음). 값 그대로 = 회원 결정 2026-11-06. 발표일 없는 보고서·상한은 원문에 없음(기본값 ①·②) |
| 종목 | KOSPI 상장 종목 | 보통주만(코드 끝자리 0, `종목거르기.보통주`), 관리·정지 다음 | 회원 결정 2026-11-06 |
| 수익률 가격 | 동일가중 수익률(가격 종류 없음) | 수정주가 S410000700(배당 없음) | 회원 결정 2026-11-06 |
| 지표 꼴 | P/B, P/E 등 | 역수 B/P, E/P 등, 높을수록 1분위 | 원문에 있음(간접 근거): p.14 "P/E와 P/OCF 지표의 경우 적자 종목 등이 많아 1분위 Long, 4분위 Short" = 적자 종목이 끝 분위(5분위)에 모이는 꼴 = 역수(E/P 음수가 맨 아래). 배수 그대로면 적자가 1분위 쪽에 섞임. 이 간접 근거는 P/E·P/OCF(적자 종목 문장)에만 해당하고, P/B·P/S·EV/EBITDA를 역수 꼴로 한 것은 같은 꼴로 넓힌 해석(원문 문장 없음) |
| 지표가 없는 종목 | — | 그 지표 값이 없거나 분모 0(±inf)이면 그 달 그 지표 분위에서 뺌. Multi는 5개 지표가 모두 있는 종목만 | 기본값 ① |
| 분위 나누기 | 1~5분위 | 종목 수를 같게(순위, 같은 값은 먼저 나온 순서). 종목이 분위 수보다 적은 날은 빈 묶음(그 달 수익률 빈칸) | 원문에 없음(기본값 ②·①) |
| 매매 가격 | 월말 리밸런싱 | 기준일(그 달 마지막 거래일) 종가에 사고팜 | 원문에 없음(기본값 ②) |
| 관리·정지 | 관리종목·거래정지 아닌 종목 | 그날 CP10001660(관리종목여부)·CP10001650(거래정지여부) 값 | 두 자료 모두 일간 |
| 상장 구간 | 상장 종목 | 시장구분(CP10000300)이 코스피·코스닥('KS'·'KQ')인 날의 가격만 계산에 씀(`읽기.상장가격`). 원본 값은 그대로 | 통합_확인기록 F2·F3. 회원 결정 2026-11-02 |
| '쓰지 말 것' 칸 | — | 리밸런싱일이 README_데이터의 '쓰지 말 것' 구간 (시작, 끝]에 드는 종목은 편입 대상에서 뺌(`종목거르기.쓰지말것`). 보유 중 걸린 칸은 빈칸(`포.쓰지말것칸`). 값은 그대로 | A008080(D-05), 수정주가 병합·감자 미반영 칸(D27). 회원 결정 2026-11-02 |
| 끝 날짜 | — | 가격·시장 일간 항목(S·CP 파일)에 값이 있는 마지막 날(`읽기.공통끝`)에서 자름 | 통합_확인기록 F6 |
| 코드 재사용 16종목 | — | 재사용 경계를 넘는 수익률을 쓰지 않음(공백 구간이 빈칸이라 끊김) | 통합_확인기록 F5, README_데이터 D-04 |
| 재무 값 뽑기 | — | 분기 보고서 = 분기(3개월) 달력기준 일간 재무의 분기말 값(`읽기.분기재무`), 사업보고서 = 결산일 값(`읽기.연간재무`). 결산월 분기는 사업보고서 값(`포.발표행`) | 일간 값은 발표 전부터 그 기간 값이 들어 있어 발표일로 자름 |
| 영업CF·EBITDA 분기 값 | — | 분기 보고서의 영업CF는 DataGuide 분기(3개월) 값: 현금흐름표는 누적 공시라 분기 3개월 값은 DataGuide 환산값. EBITDA(M000701032)는 보고서에 없는 DataGuide 계산 항목 | 계산 그대로(자료 한계) |
| 같은 회계연도 두 결산월 | — | 발표일이 늦은 보고서(같은 날이면 기간 끝이 늦은 것) | 통합_확인기록 F9(A432320) |
| 그림 29 분위 수 | "P/E 기준 1분위 Long, 3분위 Short"(p.22 주, 분위 수는 없음) | 3분위(3그룹) Q1 − Q3 | 해석: p.19가 "총 3개 그룹으로 구분해 상위 1/3 Long, 하위 1/3 Short"를 그림 25 제목에서 "1분위 Long, 3분위 Short"라 부름. 다른 읽기: 5분위로 나눈 Q1 − Q3(p.14가 적자 종목 때문에 P/E 숏을 5분위에서 4분위로 옮긴 것처럼 5분위 틀에서 숏 분위만 옮긴 것으로 읽기). 버린 이유: 그림 29 쪽(p.21 본문·p.22 그림 주)에는 분위 수 문장이 없고, 보고서에서 '1분위 Long, 3분위 Short'라는 이름이 나오는 곳은 그림 25(p.19, 위 3그룹 문장)와 그림 29뿐이라 같은 이름 = 3그룹으로 읽음(p.24 KOSDAQ도 '상위 1/3 – 하위 1/3'). 단 p.19 문장은 그림 25(애널리스트 추정치) 설명이지 그림 29 쪽 설명은 아님(원문 글자 `_검증/원문글자/02_p19.txt`·`02_p21.txt`·`02_p22.txt` 확인) |
| 시총 순위(그림 28~33·36) | 시가총액 상위 300위·100위 | 관리·정지·보통주를 거른 뒤 남은 종목에서 그날 시가총액(그 코드) 순위. 지표가 있는지 보기 전. 거래대금도 거르면 시총 먼저 | 원문에 없음(기본값 ②) |
| 현실판 벤치마크 | KOSPI 200 동일가중지수 | X01 `I.401` | 2026-11-04 고침(예전엔 I.001) |
| 현실판 기간 | 그림 36 가로축 '09.1~'18.1 | 2008-12 말 첫 리밸런싱(첫 수익률 2009-01) ~ 2018-05. I.401 대조는 2009-02부터 | '09.1 첫 점이 0 아래라 첫 수익률 달 = 2009-01로 읽음(눈대중, `원문사양_02.md` 그림 36). I.401 첫 값 2009-01-02라 2009-01 I.401 수익률은 못 냄(자료 한계). 끝 달은 원문에 없어 본 실험 끝 |
| 현실판·그림 30·31 거래대금 | 월평균 거래대금 | 기준일이 든 달의 거래일 평균 거래대금(S410000900, 원) | '월평균' 창은 원문에 없음(기본값 ②) |
| 현실판 종목 수 | Multi-Metric 동일가중(종목 수 없음) | 5분위 중 1분위(본 실험 나눔) | 원문에 없음(기본값 ②) |
| LS 누적 | 1분위 Long, k분위 Short | 월 (Q1 − Qk) 차이의 누적곱 | 원문에 없음(기본값 ②) |
| 원문 결론 | p.14~26 결론 문장 | 10절 원문 주장 3개(`02_원문주장.csv`)와 '원문 결론 판정' 11개(`02_원문결론.csv`). 그림 25·26은 G02 칸 | 원문 방향·크기만 판정. 판정 잣대 중 그림 31 감소율 비교·p.19 '300개' 백 단위 반올림은 우리가 고른 값 |


## 13. 요약표용 저장
지표별 Q1, LS, 현실판의 월수익률(원문 그림 16·18·36에 대응하는 열)을 `결과_월수익률/02_NH밸류.pkl`에 남긴다(LS는 롱숏이라 초과수익 = 수익률 그대로). 요인(F: 무위험 콜금리·시장 KOSPI I.001)은 원문에 없어 여기서 계산·저장하지 않는다(pkl의 F = None) → `연구/따라하기에서_옮김_02.ipynb` 11절이 `결과_월수익률/요인/02_NH밸류.pkl`에 저장(`평가.요약`이 F = None일 때 읽음).

In [ ]:
S = pd.DataFrame({(f"NH밸류 {n} Q1", "동일"): Q1[n] for n in Q1} | {(f"NH밸류 {n} LS", "동일"): LS[n] for n in LS}
                 | {(f"NH밸류 {k}", "동일"): o["수익률"][("동일", "Q1")] for k, o in 현실.items()})
평가.저장(S, None, "02_NH밸류", [k for k in S if k[0].endswith(" LS")])
list(S)